# Práctica de Deep Learning Multimodal CNN + Transformers

**Reducción a 2 días**
**Proyecto:** CNN + Transformer sobre DVD Rental

Este notebook implementa un sistema de recomendación multimodal utilizando tres arquitecturas de deep learning:
1. **Redes Convolucionales (CNN - ResNet50)** para extraer características visuales de pósters.
2. **Transformers (DistilBERT)** para procesar reseñas de texto.
3. **Redes Recurrentes (GRU)** para modelar secuencias temporales de preferencias de usuario.

---

## 1. Preparación y Carga de Datos
Cargamos la información de las películas y los pósters descargados.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as transforms
from torchvision import models
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image
import matplotlib.pyplot as plt

# Configuración de dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Cargar log de posters descargados
csv_path = 'data/posters/poster_download_log.csv'
if os.path.exists(csv_path):
    df_movies = pd.read_csv(csv_path)
    df_movies = df_movies[df_movies['poster_downloaded'] == True].reset_index(drop=True)
    print(f"Cargadas {len(df_movies)} películas con pósters.")
else:
    print("Error: No se encontró el log de pósters descargados.")
    df_movies = pd.DataFrame()

# Añadir reseñas simuladas para la parte de NLP
if not df_movies.empty:
    np.random.seed(42)
    reviews = [
        "Excelente película, muy recomendada con grandes actuaciones.",
        "Aburrida y predecible, no la volvería a ver.",
        "Una obra maestra visual, efectos increíbles.",
        "Buena trama pero final decepcionante.",
        "Clásico imperdible para toda la familia."
    ]
    df_movies['review'] = [np.random.choice(reviews) for _ in range(len(df_movies))]
    # Añadir género simulado para GRU
    genres = ["Action", "Comedy", "Drama", "Sci-Fi", "Horror"]
    df_movies['genre'] = [np.random.choice(genres) for _ in range(len(df_movies))]


---  
## 2. Módulo CNN (ResNet-50) + Similitud de Coseno

Utilizamos ResNet-50 preentrenada para extraer un embedding visual de 2048 dimensiones por póster.

In [ ]:
# Cargar modelo ResNet-50 preentrenado (eliminando la capa final de clasificación)
resnet = models.resnet50(pretrained=True)
# Quitar la capa FC final para obtener embeddings de 2048 dims
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet = resnet.to(device).eval()

def preprocess_image(img_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    try:
        img = Image.open(img_path).convert('RGB')
        return transform(img).unsqueeze(0).to(device)
    except:
        return None

# Generar embeddings visuales
visual_embeddings = []
valid_indices = []

print("Extrayendo embeddings visuales...")
with torch.no_grad():
    for idx, row in df_movies.iterrows():
        fname = row.get('filename')
        if not fname:
            import re
            safe_title = re.sub(r'[^\w\s-]', '', str(row['title'])).strip().replace(' ', '_')
            fname = f"{row['film_id']:04d}_{safe_title}.jpg"
        img_path = os.path.join('data/posters', fname)
        
        img_tensor = preprocess_image(img_path)
        if img_tensor is not None:
            emb = resnet(img_tensor).squeeze().cpu().numpy()
            visual_embeddings.append(emb)
            valid_indices.append(idx)
        
visual_embeddings = np.array(visual_embeddings)
df_valid = df_movies.iloc[valid_indices].reset_index(drop=True)

print(f"Embeddings visuales obtenidos: {visual_embeddings.shape}")

# Guardar embeddings
np.save('visual_embeddings.npy', visual_embeddings)

# Matriz de similitud coseno visual
sim_visual = cosine_similarity(visual_embeddings)

def recommend_visual(movie_idx, top_k=3):
    sim_scores = sim_visual[movie_idx]
    top_indices = np.argsort(sim_scores)[::-1][1:top_k+1]
    print(f"Recomendaciones visuales para '{df_valid.iloc[movie_idx]['title']}':")
    for i in top_indices:
        print(f"- {df_valid.iloc[i]['title']} (Similitud: {sim_scores[i]:.4f})")
        
if len(df_valid) > 0:
    recommend_visual(0)

---  
## 3. Módulo Transformer (DistilBERT) para Reseñas

Procesamos el texto de las reseñas para obtener representaciones semánticas de 768 dimensiones usando el token `[CLS]`.

In [ ]:
from transformers import DistilBertTokenizer, DistilBertModel

print("Cargando modelo DistilBERT...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased').to(device).eval()

text_embeddings = []

print("Extrayendo embeddings textuales...")
with torch.no_grad():
    for review in df_valid['review']:
        inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=128, padding="max_length").to(device)
        outputs = bert_model(**inputs)
        # Usar el token [CLS] (posición 0)
        cls_emb = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        text_embeddings.append(cls_emb)
        
text_embeddings = np.array(text_embeddings)
print(f"Embeddings textuales obtenidos: {text_embeddings.shape}")
np.save('text_embeddings.npy', text_embeddings)

---  
## 4. Redes Recurrentes (GRU) para Historial

Entrenamos una pequeña GRU para predecir el siguiente género basado en secuencias temporales simuladas.

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Simular secuencias de generos vistas por usuarios (Label Encoding)
genres = list(df_valid['genre'].unique())
genre_to_idx = {g: i for i, g in enumerate(genres)}
vocab_size = len(genres)

# Secuencias dummy (cada usuario vio 5 peliculas)
num_users = 100
seq_length = 4
X_seq = torch.randint(0, vocab_size, (num_users, seq_length))
y_seq = torch.randint(0, vocab_size, (num_users,))

class GRURecommender(nn.Module):
    def __init__(self, vocab_size, emb_dim=16, hidden_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.gru(x)
        out = self.fc(out[:, -1, :]) # Usar el ultimo hidden state
        return out

gru_model = GRURecommender(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(gru_model.parameters(), lr=0.01)

print("Entrenando GRU...")
for epoch in range(10):
    gru_model.train()
    optimizer.zero_grad()
    X_seq_d, y_seq_d = X_seq.to(device), y_seq.to(device)
    outputs = gru_model(X_seq_d)
    loss = criterion(outputs, y_seq_d)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

torch.save(gru_model.state_dict(), 'gru_genre.pth')
print("Modelo GRU guardado como 'gru_genre.pth'")

---  
## 5. Integración Multimodal (Fusión Tardía)

Concatenamos los embeddings visuales y textuales para una recomendación más rica.

In [ ]:
# Normalizar embeddings antes de concatenar
visual_norm = visual_embeddings / np.linalg.norm(visual_embeddings, axis=1, keepdims=True)
text_norm = text_embeddings / np.linalg.norm(text_embeddings, axis=1, keepdims=True)

# Concatenación Multimodal
combined_embeddings = np.concatenate([visual_norm, text_norm], axis=1)
sim_combined = cosine_similarity(combined_embeddings)

def recommend_multimodal(movie_idx, top_k=3):
    sim_scores = sim_combined[movie_idx]
    top_indices = np.argsort(sim_scores)[::-1][1:top_k+1]
    print(f"\nRecomendaciones Multimodales (Vision+Texto) para '{df_valid.iloc[movie_idx]['title']}':")
    for i in top_indices:
        print(f"- {df_valid.iloc[i]['title']} (Similitud Combinada: {sim_scores[i]:.4f})")

if len(df_valid) > 0:
    recommend_multimodal(0)